# Standardise LGR5 datasets

* **Developed by:** Anna Maguza
* **Affilation:** CellZome, a GSK company
* **Created date:** 2026-05-08
* **Last modified date:** 2026-05-13

Harmonise the 14 LGR5 h5ad objects into a common obs schema.


## 1. Imports and paths

In [13]:
import os, sys, gc, json, datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
sc.settings.verbosity = 2
sc.settings.dpi = 120
sc.settings.dpi_save = 300
plt.rcParams.update({
    'savefig.bbox': 'tight', 'savefig.dpi': 300, 'figure.dpi': 120,
    'font.family': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [15]:
REPO         = Path('/Users/am336941/Library/CloudStorage/OneDrive-GSK/Desktop/Fetal_stem_cells_analysis')
DATA_OUT     = Path('/Users/am336941/PhD/data/Fetal_stem_cells_analysis_enhanced')
ATLAS_PATH   = Path('/Users/am336941/PhD/data/gut_data/gut_hs_all_datasets_full_annotated_AM_30102025_181544_raw.h5ad')
LGR5_DIR     = REPO / 'data' / 'LGR5_analysis'
ORTH_PATH    = Path('/Users/am336941/PhD/data/LGR5_analysis_data/human_mouse_orthologues_ensembl.txt')
DATA_OUT.mkdir(parents=True, exist_ok=True)

In [16]:
def stamp() -> str:
    return dt.datetime.now().strftime('%d%m%Y_%H%M%S')

In [17]:
def X_is_raw(adata):
    return np.array_equal(adata.X.sum(axis=0).astype(int), adata.X.sum(axis=0))

In [18]:
LGR5_FILES = sorted(LGR5_DIR.glob('*.h5ad'))
print(f'{len(LGR5_FILES)} LGR5 h5ad files')
for f in LGR5_FILES:
    print(' -', f.name, f'{f.stat().st_size/1e6:.1f} MB')


14 LGR5 h5ad files
 - gut_hs_Ishikawa2022_AM_07052026_230357_raw.h5ad 261.9 MB
 - gut_mm_GSE135362_invivo_AM_07052026_233728_raw.h5ad 9.9 MB
 - gut_mm_GSE135362_organoid_AM_07052026_233728_raw.h5ad 15.2 MB
 - gut_mm_GSE145866_AM_07052026_233747_raw.h5ad 207.4 MB
 - gut_mm_GSE17485_AM_07052026_233113_raw.h5ad 1.8 MB
 - gut_mm_GSE23672_AM_07052026_230704_raw.h5ad 2.4 MB
 - gut_mm_GSE25109_AM_07052026_233208_raw.h5ad 2.4 MB
 - gut_mm_GSE62270_AM_07052026_233253_raw.h5ad 14.0 MB
 - gut_mm_GSE62784_AM_07052026_230514_raw.h5ad 4.0 MB
 - gut_mm_GSE73173_AM_07052026_233404_raw.h5ad 6.9 MB
 - gut_mm_GSE92865_AM_07052026_233644_raw.h5ad 389.8 MB
 - gut_mm_GSE99457_AM_07052026_233711_raw.h5ad 155.1 MB
 - gut_mm_Grün2016_AM_07052026_230009_raw.h5ad 2.6 MB
 - gut_mm_Haber2017_AM_07052026_230357_TPM.h5ad 245.9 MB


## 2. Inspect each LGR5 object

We expect each file to already carry `lgr5_status`, `lgr5_label_raw`, `technology`, `assay_modality`, `sample` (per `LGR5_data_folder_inventory.md`). This cell only inspects — no writes yet.

In [19]:
summary = []
for f in LGR5_FILES:
    a = sc.read_h5ad(f)
    row = {
        'file': f.name,
        'n_obs': a.n_obs,
        'n_vars': a.n_vars,
        'organism_guess': 'hs' if f.name.startswith('gut_hs_') else 'mm',
        'has_lgr5_status': 'lgr5_status' in a.obs.columns,
        'lgr5_status_values': sorted(a.obs['lgr5_status'].astype(str).unique()) if 'lgr5_status' in a.obs.columns else None,
        'has_technology': 'technology' in a.obs.columns,
        'assay_modality': sorted(a.obs['assay_modality'].astype(str).unique()) if 'assay_modality' in a.obs.columns else None,
        "raw_X": X_is_raw(a),
    }
    summary.append(row)
    del a
inv = pd.DataFrame(summary)
display(inv)


,file,n_obs,n_vars,organism_guess,has_lgr5_status,lgr5_status_values,has_technology,assay_modality,raw_X
0,gut_hs_Ishikawa2022_AM_07052026_230357_raw.h5ad,7103,33538,hs,True,"[LGR5+, unknown]",True,[single-cell],True
1,gut_mm_GSE135362_invivo_AM_07052026_233728_raw...,15,52636,mm,True,[LGR5+],True,[bulk],False
2,gut_mm_GSE135362_organoid_AM_07052026_233728_r...,30,52636,mm,True,[LGR5+],True,[bulk],False
3,gut_mm_GSE145866_AM_07052026_233747_raw.h5ad,4815,27998,mm,True,[LGR5+],True,[single-cell],True
4,gut_mm_GSE17485_AM_07052026_233113_raw.h5ad,2,27827,mm,True,[LGR5+],True,[microarray],False
5,gut_mm_GSE23672_AM_07052026_230704_raw.h5ad,4,27827,mm,True,"[LGR5+, LGR5-]",True,[microarray],False
6,gut_mm_GSE25109_AM_07052026_233208_raw.h5ad,4,27827,mm,True,"[LGR5+, LGR5-]",True,[microarray],False
7,gut_mm_GSE62270_AM_07052026_233253_raw.h5ad,288,23630,mm,True,[LGR5+],True,[single-cell],False
8,gut_mm_GSE62784_AM_07052026_230514_raw.h5ad,16,18315,mm,True,[LGR5+],True,[single-cell],True
9,gut_mm_GSE73173_AM_07052026_233404_raw.h5ad,12,34797,mm,True,[unknown],True,[microarray],False


## 3. Decide which datasets enter Object B (cross-species single-cell)

Single-cell mouse + Ishikawa human go in. Bulk/microarray are excluded from the integrated object and used only as signatures (notebook 1a).

In [20]:
SC_OK_MODALITIES = {'single-cell'}
inv['for_integration'] = inv['assay_modality'].apply(
    lambda v: bool(set(v or []) & SC_OK_MODALITIES)) & inv['raw_X'] == True
display(inv[['file','organism_guess','assay_modality','for_integration', "raw_X"]])
inv.to_csv(DATA_OUT / f'lgr5_inventory_{stamp()}.csv', index=False)


,file,organism_guess,assay_modality,for_integration,raw_X
0,gut_hs_Ishikawa2022_AM_07052026_230357_raw.h5ad,hs,[single-cell],True,True
1,gut_mm_GSE135362_invivo_AM_07052026_233728_raw...,mm,[bulk],False,False
2,gut_mm_GSE135362_organoid_AM_07052026_233728_r...,mm,[bulk],False,False
3,gut_mm_GSE145866_AM_07052026_233747_raw.h5ad,mm,[single-cell],True,True
4,gut_mm_GSE17485_AM_07052026_233113_raw.h5ad,mm,[microarray],False,False
5,gut_mm_GSE23672_AM_07052026_230704_raw.h5ad,mm,[microarray],False,False
6,gut_mm_GSE25109_AM_07052026_233208_raw.h5ad,mm,[microarray],False,False
7,gut_mm_GSE62270_AM_07052026_233253_raw.h5ad,mm,[single-cell],False,False
8,gut_mm_GSE62784_AM_07052026_230514_raw.h5ad,mm,[single-cell],True,True
9,gut_mm_GSE73173_AM_07052026_233404_raw.h5ad,mm,[microarray],False,False


## 4. Standardise the obs schema across single-cell LGR5 objects

All single-cell LGR5 objects get a common minimal schema for downstream concatenation:

* `study_id` ← derived from filename (`Haber2017`, `Grün2016`, `GSE92865`, …)
* `Study_name` ← long human-readable name
* `library_preparation_protocol` ← from `technology`
* `species` ← `human` / `mouse`
* `lgr5_status` ∈ `{LGR5+, LGR5-, unknown}`
* `cell_states` ← coarse epithelial state when known, else `unknown`

In [21]:
STUDY_LONG = {
    'Haber2017':   'Haber 2017 (Smart-Seq2)',
    'Grün2016':    'Grün 2016 (CEL-Seq)',
    'GSE62270':    'Grün 2015 GSE62270 (CEL-Seq)',
    'GSE62784':    'Simmini 2014 GSE62784',
    'GSE92865':    'Yan 2017 GSE92865 (10x)',
    'GSE99457':    'Yan/Kuo 2017 GSE99457 (10x)',
    'GSE145866':   'GSE145866 (10x)',
    'Ishikawa2022':'Ishikawa 2022 (10x)',
}

In [22]:
def _study_id_from_filename(name: str) -> str:
    # files look like 'gut_<sp>_<study>_AM_<ts>_<suffix>.h5ad'
    parts = name.split('_')
    return parts[2] if len(parts) >= 3 else name

In [23]:
def _normalise_var_names(a):
    if a.var_names.astype(str).str.fullmatch(r'\d+').all():
        for col in ['gene_name', 'gene_symbols', 'feature_name', 'symbol', 'Gene']:
            if col in a.var.columns:
                a.var_names = a.var[col].astype(str).values
                break
    a.var_names_make_unique()
    return a


In [24]:
for f in LGR5_FILES:
    study_id = _study_id_from_filename(f.name)
    if study_id not in STUDY_LONG:  # bulk/microarray live elsewhere
        continue
    a = sc.read_h5ad(f)
    a = _normalise_var_names(a)
    a.obs['study_id'] = study_id
    a.obs['Study_name'] = STUDY_LONG[study_id]
    a.obs['species'] = 'human' if f.name.startswith('gut_hs_') else 'mouse'
    if 'technology' in a.obs.columns and 'library_preparation_protocol' not in a.obs.columns:
        a.obs['library_preparation_protocol'] = a.obs['technology']
    if 'lgr5_status' not in a.obs.columns:
        a.obs['lgr5_status'] = 'unknown'
    if 'cell_states' not in a.obs.columns:
        a.obs['cell_states'] = 'unknown'
    _existing = a.uns.get('processing_history', {})
    proc = dict(_existing) if isinstance(_existing, dict) else {'previous': _existing}
    proc[f'standardise_obs_{stamp()}'] = {'script': '0b_lgr5_standardise.ipynb'}
    a.uns['processing_history'] = proc
    out = DATA_OUT / f'gut_{"hs" if f.name.startswith("gut_hs_") else "mm"}_{study_id}_std_{stamp()}_raw.h5ad'
    a.write_h5ad(out, compression='gzip')
    print('standardised →', out.name, a.shape)
    del a; gc.collect()

standardised → gut_hs_Ishikawa2022_std_13052026_144051_raw.h5ad (7103, 33538)
standardised → gut_mm_GSE145866_std_13052026_144053_raw.h5ad (4815, 27998)
standardised → gut_mm_GSE62270_std_13052026_144055_raw.h5ad (288, 23630)
standardised → gut_mm_GSE62784_std_13052026_144055_raw.h5ad (16, 18315)
standardised → gut_mm_GSE92865_std_13052026_144055_raw.h5ad (13247, 27998)
standardised → gut_mm_GSE99457_std_13052026_144058_raw.h5ad (5570, 27998)
standardised → gut_mm_Grün2016_std_13052026_144100_raw.h5ad (96, 57186)
standardised → gut_mm_Haber2017_std_13052026_144100_raw.h5ad (1522, 20107)
